
# 🧬 Feature Selection & Data Profiling

**Objective:** Analyze feature importance, reduce dimensionality using MRMR (Minimum Redundancy Maximum Relevance) via `FeatureWiz`, and prepare the final dataset for the elasticity model.
**Input:** `finance_db.gold_sales_events_enriched`
**Output:** Selected feature list and processed dataframe logic.


## 📚 1. Libraries

In [ ]:
import sys
display({"python_version": sys.version})

In [ ]:
# Install Feature Selection library
!pip install featurewiz
dbutils.library.restartPython()

In [ ]:
# Install EDA library
!pip install sweetviz
dbutils.library.restartPython()

In [ ]:
# Ensure TensorFlow compatibility
!pip install "tensorflow>=2.5"
dbutils.library.restartPython()

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import zscore
from scipy import stats
import patsy
import os
import re

# Spark
from pyspark.sql.functions import col, count, when, lit, sum as spark_sum
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import DataFrame

# Feature Selection
from featurewiz import FeatureWiz

# Visualization Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid", palette="pastel")
import warnings
warnings.filterwarnings('ignore')

# Disable MLflow autolog for this step to avoid clutter
mlflow.autolog(disable=True)


## 📥 2. Data Loading & Pre-processing

In [ ]:
# Load enriched data from Gold layer
df_data = spark.table("finance_db.gold_sales_events_enriched")

# Casting types for consistency
df_data = df_data.withColumn(
    "week_of_year", 
    col("week_of_year").cast('integer')).withColumn(
    "day_of_week", 
    col("day_of_week").cast('integer')).withColumn(
    "THEORETICAL_UNIT_MARGIN", 
    col("THEORETICAL_UNIT_MARGIN").cast('double'))

In [ ]:
def group_rare_categories(df, cat_cols, threshold=0.80, volume_col='Q26'):
    """
    Groups rare categories in specified columns based on cumulative coverage.

    For each categorical column:
      1. Calculates counts and proportions.
      2. Sorts by descending proportion and calculates cumulative sum.
      3. Keeps categories that explain up to `threshold` of the data.
      4. Groups the rest into 'others'.
    
    Creates new columns with suffix '_proc'.

    Args:
        df (pyspark.sql.DataFrame): Input DataFrame.
        cat_cols (list): List of categorical column names.
        threshold (float): Cumulative coverage threshold (default 0.80).
        volume_col (str): Volume column for Z-score calculation.

    Returns:
        pyspark.sql.DataFrame: Processed DataFrame.
    """
    total_rows = df.count()
    df_proc = df

    for col_name in cat_cols:
        # 1) Calculate absolute count and frequency
        freq_df = (
            df
            .groupBy(col_name)
            .agg(count("*").alias("count"))
            .withColumn("freq", F.col("count") / F.lit(total_rows)))
        # 2) Sort and calculate cumulative proportion
        w = Window.orderBy(F.desc("freq"))
        freq_cum = freq_df.withColumn("cum_prop", F.sum("freq").over(w))

        # 3) Select categories within threshold
        categories_to_keep = (
            freq_cum
            .filter(F.col("cum_prop") <= threshold)
            .select(col_name)
            .rdd.flatMap(lambda x: x)
            .collect())

        # 4) Create processed column grouping remainder as 'others'
        col_name_proc = f"{col_name}_proc"
        df_proc = df_proc.withColumn(
            col_name_proc,
            when(col(col_name).isin(categories_to_keep), col(col_name))
            .otherwise("others"))

    #---------------------------------------------------------------
    # Specific Logic: Brand Category Reduction
    #---------------------------------------------------------------
    # Binary flag for main brand categories (assumed logic: IDs <= 8 are main)
    df_proc = df_proc.withColumn(
        'BRAND_CATEGORY_reduced',
        F.when(F.col('BRAND_CATEGORY').cast('int') <= 8, F.lit(1)).otherwise(F.lit(0)))

    #---------------------------------------------------------------
    # Z-Score Calculation for Target Volume
    #---------------------------------------------------------------
    stats_row = df_proc.select(
        F.mean(volume_col).alias('mean_v'),
        F.stddev(volume_col).alias('std_v')).collect()[0]

    z_col = f"{volume_col}_zscore"
    df_proc = df_proc.withColumn(
        z_col,
        (F.col(volume_col) - F.lit(stats_row['mean_v'])) / F.lit(stats_row['std_v']))

    return df_proc

In [ ]:
# Define categorical variables to process
categorical_columns = ['BRAND',
    'SUB_BU',
    'REGION_STATE',
    'SUB_CHANNEL',
    'SALES_TEAM_CHANNEL',
    'month',
    'EAN_CODE']

# Apply rare category grouping
df_data = group_rare_categories(
    df_data, 
    categorical_columns, 
    threshold=0.80, 
    volume_col='BILLED_QTY')

In [ ]:
import math

def prepare_modeling_df(df: DataFrame, volume_col: str = 'Q26', cat_vars: list = None, log_Q = False) -> DataFrame:
    """
    Prepares the DataFrame for modeling:
      1. Creates target 'Q' from volume column.
      2. Log-transforms price ('ln_price').
      3. Creates cyclic features (sin_w, cos_w) from week_of_year.
      4. Casts categorical variables to string.
    """
    # Transformations
    df2 = df.withColumn('ln_price', F.log(F.col('NET_PRICE_UNIT'))) \
            .withColumn(
                'sin_w',
                F.sin(2 * F.lit(math.pi) * (F.col('week_of_year') - 1) / 52)) \
            .withColumn(
                'cos_w',
                F.cos(2 * F.lit(math.pi) * (F.col('week_of_year') - 1) / 52))

    if log_Q:
        df2 = df2.withColumn('Q', F.log(F.col(volume_col)))
    else:
        df2 = df2.withColumn('Q', F.col(volume_col))

    # Cast Categoricals
    if cat_vars:
        for c in cat_vars:
            df2 = df2.withColumn(c, F.col(c).cast('string'))

    # Select Final Columns
    cols_final = ['Q', 'ln_price', 'sin_w', 'cos_w'] + (cat_vars if cat_vars else [])
    model_df = df2.select(*cols_final)
    
    return model_df

In [ ]:
# List of all potential variables for the model
cat_vars_list = ['EAN_CODE_proc',
    'EAN_CODE', 
    'BRAND_proc',
    'SUB_BU_proc',
    'REGION_STATE_proc',
    'SUB_CHANNEL_proc',
    'BRAND',
    'REGION_STATE',
    'REGION', 
    'BUSINESS_UNIT',
    'SUB_BU', 
    'BRAND_CATEGORY',
    'SUB_CHANNEL', 
    'SALES_TEAM_CHANNEL', 
    'SALES_TEAM_CHANNEL_proc',
    'TEAM_NAME',
    'month', 
    'month_proc',
    'day_of_week',
    'price_change',
    'week_of_year',
    'BRAND_CATEGORY_reduced',
    'THEORETICAL_UNIT_MARGIN']

model_df = prepare_modeling_df(df_data, volume_col='BILLED_QTY', cat_vars=cat_vars_list, log_Q=False)
model_df = model_df.withColumn('week_of_year', F.col('week_of_year').cast('int'))

In [ ]:
# Validate dimensionality reduction (Original vs Processed Categories)
processed_cols = [c for c in cat_vars_list if c.endswith('_proc') or c.endswith('_reduced')]
original_cols = set(cat_vars_list) - set(processed_cols)

results = []
for col_proc in processed_cols:
    if col_proc.endswith('_proc'):
        col_orig = col_proc.removesuffix('_proc')
    elif col_proc.endswith('_reduced'):
        col_orig = col_proc.removesuffix('_reduced')
    else:
        continue

    if col_orig in original_cols:
        count_orig = df_data.select(col_orig).distinct().count()
        count_proc = df_data.select(col_proc).distinct().count()
        results.append((col_orig, count_orig, col_proc, count_proc))

display(spark.createDataFrame(
    results,
    ["original_column", "original_cardinality", "processed_column", "processed_cardinality"]))


## 📊 3. Feature Selection (Preparation)

In [ ]:
target_col = ['Q']

num_vars = [
    "ln_price",
    "sin_w",
    "cos_w",
    "week_of_year"]

cat_vars_final = [
    # Using mostly processed variables to reduce cardinality
    "day_of_week",
    "BRAND_proc",
    "REGION_STATE_proc",
    "SUB_CHANNEL_proc",
    "SALES_TEAM_CHANNEL_proc",
    "month_proc",
    "BRAND_CATEGORY_reduced",
    "BUSINESS_UNIT",
    "SUB_BU",
    "REGION"]

all_cols_sel = target_col + num_vars + cat_vars_final

# Filter Outliers and Sample Data for Feature Selection (Performance Optimization)
# We keep 80% of volume distribution and take a sample of 1M rows
percentile_val = model_df.approxQuantile('Q', [0.80], 0.01)[0]

df_pandas = (
    model_df
    .select(all_cols_sel)
    .filter((F.col('Q') >= 1) & (F.col('Q') <= percentile_val))
    .orderBy(F.rand())
    .limit(1000000)
    .toPandas())

# Adjust Data Types for Python libraries
df_pandas[target_col] = df_pandas[target_col].astype(int)
df_pandas[cat_vars_final] = df_pandas[cat_vars_final].astype('str')

cols_float = ['ln_price', 'sin_w', 'cos_w']
df_pandas[cols_float] = df_pandas[cols_float].astype('float')
df_pandas['week_of_year'] = df_pandas['week_of_year'].astype('int')

# Log-transform target for analysis stability (Log(Q + 1))
df_pandas[target_col[0]] = np.log(df_pandas[target_col[0]] + 1)
df_pandas['Q'] = df_pandas['Q'].astype('float')

df_pandas.info()


### 3.1 Exploratory Analysis (Sweetviz)

In [ ]:
import sweetviz as sv

# Force Target as Numerical
feature_config = sv.FeatureConfig(force_num=["Q"])

report = sv.analyze(
    df_pandas,
    target_feat='Q',
    feat_cfg=feature_config)

report_path = '/dbfs/mnt/sandbox/reports/data_profile_for_modeling.html'
report.show_html(filepath=report_path, open_browser=False)

displayHTML(f"""
  <a href="files/sandbox/reports/data_profile_for_modeling.html"
     download="data_profile_for_modeling.html"
     style="font-size:16px;">
    ⬇️ Download Sweetviz Report (HTML)
  </a>
""")

In [ ]:
from sweetviz.sv_types import FeatureType

def associations_to_dataframe(report) -> pd.DataFrame:
    """
    Extracts association metrics from Sweetviz report into a long-form DataFrame.
    Identifies metric types: Pearson (num-num), Uncertainty (cat-cat), Correlation Ratio (cat-num).
    """
    rows = []
    for feat, assoc_dict in report._associations.items():
        for other, val in assoc_dict.items():
            t1 = report.get_type(feat)
            t2 = report.get_type(other)
            
            if t1 == FeatureType.TYPE_NUM and t2 == FeatureType.TYPE_NUM:
                assoc_type = "pearson"
            elif t1 in (FeatureType.TYPE_CAT, FeatureType.TYPE_BOOL) and \
                 t2 in (FeatureType.TYPE_CAT, FeatureType.TYPE_BOOL):
                assoc_type = "uncertainty"         # Theil’s U (asymmetric)
            else:
                assoc_type = "correlation_ratio"   # eta-squared (asymmetric)
                
            rows.append({
                "feature_1": feat,
                "feature_2": other,
                "association_type": assoc_type,
                "association_value": val
            })
    return pd.DataFrame(rows)

# Generate DataFrames
df_assoc = associations_to_dataframe(report)

display(df_assoc)


### 3.2 Feature Selection (FeatureWiz)
Using MRMR (Minimum Redundancy Maximum Relevance) and Recursive XGBoost.

In [ ]:
# 1. Separate X and y
X = df_pandas.drop(columns=target_col)
y = df_pandas[target_col]

# 2. Run FeatureWiz (Run 1: Without One-Hot Encoding to test raw power)
# FeatureWiz automatically handles high cardinality if one-hot is not forced
fwiz_no_onehot = FeatureWiz(
    corr_limit=0.75,                # SULOV correlation limit
    verbose=2, 
    feature_engg='', 
    category_encoders='',           # Let algorithm decide or keep label encoded
    dask_xgboost_flag=False,
    skip_sulov=False,
    skip_xgboost=False)

X_selected, y_selected = fwiz_no_onehot.fit_transform(X, y)

In [ ]:
print("Features selected (No One-Hot):")
for c in fwiz_no_onehot.features:
  print(c)

In [ ]:
# 3. Run FeatureWiz (Run 2: Forcing One-Hot Encoding)
# This helps identify if specific levels of categorical variables are strong drivers
fwiz_onehot = FeatureWiz(
    corr_limit=0.75,
    verbose=2,
    feature_engg='',
    category_encoders=['onehot', 'onehot'], # Force one-hot
    dask_xgboost_flag=False,
    skip_sulov=False,
    skip_xgboost=False)

X_selected_onehot, y_selected_onehot = fwiz_onehot.fit_transform(X, y)


### 3.3 Selection Logic & Post-processing
FeatureWiz with OneHot returns feature names like `BRAND_BrandName`. We need to map these back to the original variables to decide which full categorical variables to keep in the model.

In [ ]:
def clean_category_name(name: str) -> str:
    """
    Mimics FeatureWiz category name cleaning: uppercases and removes non-alphanumeric chars.
    """
    if not isinstance(name, str):
        name = str(name)
    return re.sub(r'[^A-Z0-9]', '', name.upper())

# Analyze which categorical levels were kept
selected_features = fwiz_onehot.features
cat_analysis = {}

print("Analyzing selected categorical features...")

for var in cat_vars_final:
    # 1. Get original unique levels from data
    original_levels = set(df_pandas[var].unique())
    total_original = len(original_levels)

    # 2. Map cleaned names back to original names
    clean_to_original_map = {
        clean_category_name(level): level for level in original_levels
    }

    # 3. Extract levels kept by FeatureWiz for this variable
    # FeatureWiz format is usually "VAR_LEVEL"
    prefix = var + '_'
    kept_clean_levels = {
        f.replace(prefix, '') for f in selected_features if f.startswith(prefix)
    }

    # 4. Translate back to original names
    kept_original_levels = set()
    for clean_name in kept_clean_levels:
        if clean_name in clean_to_original_map:
            kept_original_levels.add(clean_to_original_map[clean_name])

    removed_original_levels = original_levels - kept_original_levels

    if total_original > 0:
        percent_kept = (len(kept_original_levels) / total_original) * 100
        cat_analysis[var] = {
            'total': total_original,
            'n_kept': len(kept_original_levels),
            'percent_kept': percent_kept,
            'levels_kept': sorted(list(kept_original_levels)),
            'levels_removed': sorted(list(removed_original_levels)),
        }

# Display Results
for var, analysis in cat_analysis.items():
    print("-" * 80)
    print(f"🔎 Variable: '{var}'")
    print(f"  - Kept Levels: {analysis['n_kept']} / {analysis['total']} ({analysis['percent_kept']:.2f}%)")
    # print(f"  - Levels Kept: {analysis['levels_kept']}") # Uncomment for details

In [ ]:
# Final Feature Selection Logic
threshold_percent = 50.0  # e.g., keep variable if > 50% of its levels are significant (or custom logic)

# 1. Numeric features kept by FeatureWiz
numeric_kept = [f for f in selected_features if f in num_vars]

# 2. Categorical features kept based on analysis threshold
categorical_kept = [
    var for var, analysis in cat_analysis.items()
    if analysis['percent_kept'] >= threshold_percent 
    # OR manual override logic can be added here]

# Note: Sometimes we keep a variable even if few levels are selected, 
# but we group the non-selected ones into 'others'.

final_features = numeric_kept + categorical_kept

print("\n" + "=" * 80)
print(f"📊 Final Selection (Threshold: {threshold_percent}%)")
print("=" * 80)
print(f"Numeric: {numeric_kept}")
print(f"Categorical: {categorical_kept}")

In [ ]:
def group_removed_categories(df_input: pd.DataFrame, analysis_dict: dict) -> pd.DataFrame:
    """
    Groups categories identified as 'removed' (non-significant) into 'others'.
    Only applies if at least one category was significant for that variable.
    """
    df_proc = df_input.copy()
    
    print("Grouping removed categories into 'others'...")
    
    for var_name, analysis in analysis_dict.items():
        levels_removed = analysis.get('levels_removed')
        n_kept = analysis.get('n_kept', 0)

        if var_name in df_proc.columns and levels_removed and n_kept > 0:
            print(f"-> Processing '{var_name}': Grouping {len(levels_removed)} levels.")
            mask = df_proc[var_name].isin(levels_removed)
            df_proc.loc[mask, var_name] = 'others'
            
        elif var_name in df_proc.columns:
             print(f"-> Ignoring '{var_name}': No levels were significant (n_kept=0).")
            
    return df_proc

In [ ]:
# Apply grouping logic to the sample dataframe for validation
df_final_sample = group_removed_categories(df_pandas, cat_analysis)
display(df_final_sample.head())